# 01 — Exploratory Data Analysis

**Tujuan.** Memahami struktur 9 tabel train + 9 tabel test, memeriksa kualitas data
(missing value, duplikasi, integritas relasi), dan menemukan sinyal yang berkaitan dengan
target `status_penanganan` sebelum menyusun strategi pemodelan.

**Data.** 9 tabel relasional per split: `shipments` (tabel utama, 1 baris = 1 pengiriman),
`scan_events`, `route_segments`, `invoices`, serta 5 tabel master (`customers`, `couriers`,
`hubs`, `service_packages`, `promotions`).

**Metrik kompetisi.** Macro F1-score atas 7 kelas target — karena setiap kelas dihitung
setara, kelas minoritas (`klaim_hilang`, `rusak_perjalanan`) tidak boleh diabaikan.

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
%matplotlib inline

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

# root proyek = folder terdekat ke atas yang berisi src/ dan data/
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(BASE))

from src.data_loader import (BASE_DIR, RAW_DIR, TRAIN_DIR, TEST_DIR, TARGET, CLASSES,
                             TABLES, path_of, table_overview, SAMPLE_SUBMISSION)
from src.utils import set_seed

set_seed()
print("BASE_DIR:", BASE_DIR)
print("pandas  :", pd.__version__, "| polars:", pl.__version__, "| numpy:", np.__version__)

## 1. Ukuran tabel

Seluruh 18 CSV dibaca dengan polars `scan_csv` (lazy) sehingga hanya kolom yang dipakai
yang dimaterialisasi — penting karena RAM mesin hanya 8 GB.

In [ ]:
info = table_overview()
pivot = (info.pivot(index="table", columns="split", values=["rows", "cols", "size_mb"])
             .reindex(TABLES))
display(pivot)
print("total baris shipments (train+test):", info.query("table == 'shipments'")["rows"].sum())

## 2. Skema kolom tiap tabel

In [ ]:
for t in TABLES:
    schema = pl.scan_csv(path_of("train", t), infer_schema_length=5000).collect_schema()
    print(f"--- {t}  ({len(schema)} kolom)")
    print(", ".join(f"{k}:{str(v).replace('String', 'str')}" for k, v in schema.items()))
    print()

## 3. Kualitas data: missing value

Tabel `shipments` adalah tabel utama; tabel anak bersifat *one-to-many* sehingga tidak
boleh langsung di-join tanpa agregasi terlebih dahulu.

In [ ]:
quality = []
for t in TABLES:
    lf = pl.scan_csv(path_of("train", t), infer_schema_length=5000)
    cols = lf.collect_schema().names()
    n = lf.select(pl.len()).collect().item()
    agg = lf.select([pl.col(c).null_count().alias(c) for c in cols]).collect().row(0, named=True)
    miss = {c: agg[c] / n for c in cols if agg[c] > 0}
    worst = sorted(miss.items(), key=lambda x: -x[1])[:4]
    quality.append({
        "table": t, "rows": n, "cols": len(cols), "kolom_missing": len(miss),
        "missing_terbesar": "; ".join(f"{c} {v:.1%}" for c, v in worst) if worst else "-",
    })

display(pd.DataFrame(quality))

## 4. Integritas relasi train ↔ test

Pemeriksaan: (a) `shipment_id` unik, (b) tidak ada irisan id train-test (kalau ada,
artinya test bocor ke train), (c) setiap baris tabel anak menunjuk `shipment_id` yang ada,
(d) id pada `sample_submission.csv` sama persis dengan id test.

In [ ]:
def id_set(split, table, col="shipment_id"):
    return set(pl.scan_csv(path_of(split, table), infer_schema_length=5000)
               .select(pl.col(col).unique()).collect()[col].to_list())

sh_tr, sh_te = id_set("train", "shipments"), id_set("test", "shipments")
print(f"shipments unik    : train={len(sh_tr):,}  test={len(sh_te):,}")
print(f"overlap train-test: {len(sh_tr & sh_te):,}  (harus 0)")

for t in ["scan_events", "route_segments", "invoices"]:
    a, b = id_set("train", t), id_set("test", t)
    orphan = len(a - sh_tr) + len(b - sh_te)
    print(f"{t:15s} cakupan train={len(a & sh_tr)/len(sh_tr):.1%} "
          f"test={len(b & sh_te)/len(sh_te):.1%} orphan={orphan}")

sub = pd.read_csv(SAMPLE_SUBMISSION)
print("\nsample_submission:", sub.shape, list(sub.columns))
print("id sample == id test shipments:", set(sub.shipment_id) == sh_te)
print("contoh label pada sample      :", sub.status_penanganan.unique()[:3])

## 5. Distribusi target

Target tidak seimbang: `normal` mendominasi, sedangkan `klaim_hilang` dan
`rusak_perjalanan` berukuran kecil. Karena metrik resmi adalah **macro F1**, kelas
minoritas tetap berbobot sama dengan `normal` → perlu bobot kelas / penyesuaian ambang.

In [ ]:
dist = (pl.scan_csv(TRAIN_DIR / "shipments.csv", infer_schema_length=5000)
          .group_by(TARGET).agg(pl.len().alias("n"))
          .with_columns((pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct"))
          .sort("n", descending=True).collect().to_pandas())
display(dist)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dist[TARGET], dist["n"], color="#3b6ea5")
ax.set_title("Distribusi status_penanganan (train)")
ax.set_ylabel("jumlah pengiriman")
for i, (v, p) in enumerate(zip(dist["n"], dist["pct"])):
    ax.text(i, v, f"{p:.1f}%", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.show()

print("rasio kelas terbanyak : terkecil =", round(dist.n.max() / dist.n.min(), 1))

## 6. Fitur numerik — perbandingan train vs test

In [ ]:
NUM_COLS = ["weight_kg", "volume_l", "declared_value", "distance_km", "order_to_ship_hours",
            "berat_volumetrik", "qty_items", "jumlah_catatan", "coupon_count", "ship_hour"]

def num_stats(split):
    lf = pl.scan_csv(path_of(split, "shipments"), infer_schema_length=5000)
    aggs = []
    for c in NUM_COLS:
        aggs += [pl.col(c).mean().alias(f"{c}|mean"), pl.col(c).std().alias(f"{c}|std"),
                 pl.col(c).median().alias(f"{c}|median"), pl.col(c).max().alias(f"{c}|max"),
                 pl.col(c).null_count().alias(f"{c}|null")]
    d = lf.select(aggs).collect().row(0, named=True)
    out = pd.DataFrame({c: {k: d[f"{c}|{k}"] for k in ["mean", "std", "median", "max", "null"]}
                        for c in NUM_COLS}).T
    out["null"] = (out["null"] / lf.select(pl.len()).collect().item()).round(4)
    return out.round(3)

cmp = num_stats("train").add_suffix("_train").join(num_stats("test").add_suffix("_test"))
display(cmp[["mean_train", "mean_test", "std_train", "std_test",
             "median_train", "median_test", "max_train", "null_train"]])

## 7. Fitur kategorikal vs target

In [ ]:
CAT_EDA = ["channel", "goods_category", "dest_region", "prioritas_pengiriman"]

df = (pl.scan_csv(TRAIN_DIR / "shipments.csv", infer_schema_length=5000)
        .select(CAT_EDA + [TARGET]).collect().to_pandas())

for c in ["goods_category", "dest_region", "prioritas_pengiriman"]:
    t = pd.crosstab(df[c], df[TARGET], normalize="index").round(3)
    t["n"] = df[c].value_counts()
    print(f"--- {c} (proporsi kelas per kategori)")
    display(t.sort_values("n", ascending=False))

## 8. Sinyal dari tabel anak: `scan_events`

Jumlah scan, ada/tidaknya event `selesai`, jumlah upaya antar, dan jeda antar scan
(`gap_hours`) diduga kuat membedakan pengiriman normal dari yang tertunda / gagal antar.

In [ ]:
sc = (pl.scan_csv(TRAIN_DIR / "scan_events.csv", infer_schema_length=5000)
      .group_by("shipment_id")
      .agg([pl.len().alias("n_scan"),
            (pl.col("event_type") == "selesai").sum().alias("n_selesai"),
            (pl.col("event_type") == "upaya_antar").sum().alias("n_upaya"),
            pl.col("gap_hours").mean().alias("gap_mean"),
            pl.col("gap_hours").max().alias("gap_max"),
            pl.col("scan_quality").mean().alias("q_mean")])
      .collect())

tgt = (pl.scan_csv(TRAIN_DIR / "shipments.csv", infer_schema_length=5000)
         .select(["shipment_id", TARGET]).collect())
sc = sc.join(tgt, on="shipment_id", how="inner").to_pandas()
sc["no_selesai"] = (sc["n_selesai"] == 0).astype(int)

agg = (sc.groupby(TARGET)[["n_scan", "n_upaya", "n_selesai", "no_selesai",
                           "gap_mean", "gap_max", "q_mean"]].mean()
         .reindex(CLASSES).round(3))
agg["n"] = sc[TARGET].value_counts().reindex(CLASSES)
display(agg)

## 9. Sinyal dari tabel anak: `invoices`

In [ ]:
inv = (pl.scan_csv(TRAIN_DIR / "invoices.csv", infer_schema_length=5000)
       .group_by("shipment_id")
       .agg([(pl.col("invoice_type") == "klaim").sum().alias("n_klaim_inv"),
             (pl.col("payment_status") != "lunas").sum().alias("n_pay_problem"),
             pl.col("amount").sum().alias("amt_sum")])
       .collect())
inv = inv.join(tgt, on="shipment_id", how="inner").to_pandas()
inv["ada_invoice_klaim"] = (inv["n_klaim_inv"] > 0).astype(int)
inv["ada_pay_problem"] = (inv["n_pay_problem"] > 0).astype(int)

rate = (inv.groupby(TARGET)[["ada_invoice_klaim", "ada_pay_problem"]].mean()
           .reindex(CLASSES).round(3))
rate["n"] = inv[TARGET].value_counts().reindex(CLASSES)
rate.columns = ["rate_invoice_klaim", "rate_pay_bermasalah", "n"]
display(rate)

print("\nDistribusi kelas untuk pengiriman yang punya invoice klaim:")
print(inv.loc[inv.ada_invoice_klaim == 1, TARGET].value_counts(normalize=True).round(3))

## 10. Insight EDA → keputusan strategi

1. **Grain data.** `shipments` satu baris per pengiriman; tabel anak harus diagregasi ke
   level `shipment_id` sebelum join agar tidak menggandakan baris.
2. **Tabel anak informatif.** Varian `event_type`, `scan_quality`, `gap_hours`, jenis
   invoice (`klaim`), dan status bayar memberi sinyal yang tidak ada di tabel utama.
3. **Imbalance.** Karena macro F1, strategi wajib menyeimbangkan kelas: bobot kelas pada
   LightGBM + penyesuaian pengali probabilitas saat prediksi.
4. **Tidak ada kebocoran train→test.** Tidak ada irisan `shipment_id`; fitur turunan harus
   dihitung per split, dan transformasi berbasis target (target encoding) harus out-of-fold.
5. **Distribusi train ≈ test.** Statistik numerik dan kategori serupa → validasi hold-out
   representatif.
6. **Kardinalitas tinggi.** `courier_id` (±24 ribu), `customer_id` (±539 ribu), `hub_id`
   (654) terlalu besar untuk one-hot → dipakai target encoding out-of-fold.